<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 — Exercise: Build Your First LangGraph

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

- Describe a graph's **state** with a single `TypedDict` class
- Write a **node** — a plain function that takes the state and returns only what it changed
- Wire nodes together with **edges**, from `START` to `END`
- **Compile**, **draw** and **run** the graph, and read the state that comes back

```
START → validate_marks → calculate_total → assign_grade → write_summary → END
```

One state class · four nodes · five edges. A student result pipeline.

> **No API key, no cost.** There is no LLM in this notebook on purpose — a graph is
> just functions and arrows, and that is easier to see when no model is in the way.

Each step gives you the **instructions as comments** — you write the code underneath.
Run a cell with **Shift + Enter**.

---

In [ ]:
# --- Setup: run this cell first (~20 seconds) --------------------------------
# LangGraph is the only install we need. Everything else is standard Python.

!pip install -q langgraph

# TypedDict is how we describe the SHAPE of the state (which keys, which types).
from typing import TypedDict

# StateGraph builds the graph. START and END are the two built-in markers
# for where a run begins and where it finishes.
from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

print("ready")

---

## The three words

LangGraph is built from exactly three ideas. Keep these on screen while you work.

| Word | What it is | In this notebook |
| --- | --- | --- |
| **State** | one dictionary that flows through the graph | `StudentState` — name, marks, total, grade, summary |
| **Node** | a function: takes the state, returns an **update** to it | `validate_marks`, `calculate_total`, `assign_grade`, `write_summary` |
| **Edge** | what runs next | a straight line from `START` to `END` |

**The two rules of state:**
1. A node **receives the whole state**.
2. A node **returns only the keys it wants to change** — LangGraph merges the rest for you.

---

## Step 1 — Define the state

One class. Every node reads from it, and each node fills in exactly one field.

In [ ]:
# STEP 1 - Define the state
#
# 1. create a class called StudentState that inherits from TypedDict
# 2. give it five fields, with a comment saying who fills each one in:
#       name     : str    <- you supply this when you start the graph
#       marks    : list   <- you supply this when you start the graph
#       total    : int    <- calculate_total will write it
#       grade    : str    <- assign_grade will write it
#       summary  : str    <- write_summary will write it
# 3. print(StudentState.__annotations__) to check the shape came out right

## Step 2 — The first two nodes

A node is an ordinary Python function. Nothing about it is special until you add it to a graph.

In [ ]:
# STEP 2 - Write the first two nodes
#
# Node 1 - validate_marks(state)
#   1. print("  [node] validate_marks")   so you can watch it run later
#   2. keep only the marks between 0 and 100, and drop anything else
#   3. return {"marks": <the cleaned list>}
#
# Node 2 - calculate_total(state)
#   1. print("  [node] calculate_total")
#   2. add up state["marks"]
#   3. return {"total": <the sum>}
#
# Notice: each one returns a dict with ONE key. Never the whole state.

## Step 3 — The last two nodes

Same pattern. `assign_grade` reads what `calculate_total` wrote — that is the state doing its job.

In [ ]:
# STEP 3 - Write the last two nodes
#
# Node 3 - assign_grade(state)
#   1. print("  [node] assign_grade")
#   2. work out the average from state["total"] and len(state["marks"])
#   3. "A" if the average is 80 or more, "B" from 60, "C" from 40, otherwise "F"
#   4. return {"grade": <the letter>}
#
# Node 4 - write_summary(state)
#   1. print("  [node] write_summary")
#   2. build ONE sentence using state["name"], state["total"] and state["grade"]
#   3. return {"summary": <the sentence>}
#
# Node 3 reads state["total"] - a key it never set itself. That value is there
# because calculate_total returned it and LangGraph merged it in.

## Step 4 — Build the graph

Four nodes, five edges. The edges are what make this a graph instead of four loose functions.

In [ ]:
# STEP 4 - Build the graph
#
# 1. builder = StateGraph(StudentState)          <- the state class goes here
#
# 2. add all four nodes. The first argument is the NAME (a string you choose),
#    the second is the function itself:
#       builder.add_node("validate_marks", validate_marks)
#    ...and the same for calculate_total, assign_grade, write_summary
#
# 3. wire five edges in a straight line:
#       START -> validate_marks -> calculate_total -> assign_grade -> write_summary -> END
#    each one is builder.add_edge("from", "to")
#    START and END are objects, not strings - pass them without quotes
#
# 4. graph = builder.compile()
#
# 5. print("compiled") so you know it worked

## Step 5 — Draw it

This cell is written for you. Check the picture matches the line you wired in Step 4.

In [ ]:
# Given - just run it.
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # the PNG is rendered online, so this covers a slow or blocked network
    print("(diagram unavailable - here are the edges instead)")
    print("\n".join(f"  {e.source} -> {e.target}" for e in graph.get_graph().edges))

## Step 6 — Run it

Watch the four `[node]` lines print in order, then look at what comes back.

In [ ]:
# STEP 6 - Run the graph
#
# 1. call graph.invoke({...}) with a starting state. You only supply the fields
#    you actually know at the start:
#       {"name": "Aditi", "marks": [88, 91, 79, 95]}
# 2. store what comes back in a variable called result
# 3. print the WHOLE result dictionary and read it carefully
# 4. then print just result["summary"]
#
# Question to answer for yourself: you passed in two keys. How many came back?

## Step 7 — Change one thing, run again

Same graph, different input. Nothing about the graph needs touching.

In [ ]:
# STEP 7 - Change the input and re-run
#
# 1. put a weaker set of marks in a variable, for example [35, 42, 28, 51]
# 2. invoke the graph again with the same name and the new marks
# 3. print the grade - it should be different from Step 6
#
# 4. now sneak an impossible mark into the list, e.g. [35, 42, 150, 51]
# 5. run it again and print result["marks"]
#    Did validate_marks drop the 150? What did that do to the total -
#    and to the grade? Is that the behaviour you would want in a real system?

---

## Stretch — try these on your own

No steps for these. Work them out from what you built above.

**A. Add a fifth node.** Insert an `add_remark` node **between** `assign_grade` and
`write_summary` that writes a `remark` field — something like "Distinction" for an A and
"Needs improvement" for an F. You will need to change the state class, add the node and
re-wire two edges. Draw the graph again and check the new box sits where you expect.

**B. Prove the merge rule.** Make `calculate_total` return `{}` instead of a total, then
run the graph. What happens, and why? Write one sentence explaining it.

**C. Break it on purpose.** Add an edge from `write_summary` back to `validate_marks` and
run it. Read what happens, then explain in one line what that tells you about edges.

---

### ✅ What you practised

| Idea | The one-liner |
| --- | --- |
| **State** | one `TypedDict` class describing every key that flows through the graph |
| **Node** | a plain function — takes the whole state, returns only the keys it changed |
| **Edge** | `builder.add_edge("from", "to")` — what runs next |
| **`START` / `END`** | the built-in entry and exit markers; pass them without quotes |
| **`compile()`** | turns the builder into a runnable graph |
| **`invoke()`** | runs it — you pass the keys you know, you get every key back |
| **The merge** | you returned one key per node and got a full dictionary back; LangGraph merged the updates as it went |

> **Next:** in the class notebook the edges stop being a straight line. A **conditional
> edge** lets the graph pick which node runs next — that is routing, and it is the
> difference between a pipeline and something that makes a decision.